# Notebook 1: Compound data acquisition from ChEMBL
**Target:** Androgen Receptor (CHEMBL1871)

Our objective is to extract binding data for the human Androgen Receptor (AR) from the ChEMBL database. We will pull all available biological activity records and filter strictly for `IC50` assays to ensure data consistency for our Machine Learning model.

In [ ]:
import math
from pathlib import Path
from zipfile import ZipFile
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from rdkit.Chem import PandasTools
from chembl_webresource_client.new_client import new_client
from tqdm.auto import tqdm

In [ ]:
HERE = Path.cwd()
DATA = HERE / "data"

### 1. Data Acquisition from ChEMBL
Our objective is to extract binding data for the human Androgen Receptor (AR), a primary driver of prostate cancer growth. We will query the ChEMBL database using its UniProt ID (`P10275`) and pull all available biological activity records. To ensure data consistency, we will only fetch `IC50` assays.

In [ ]:
# Create resource objects for API access
targets_api = new_client.target
compounds_api = new_client.molecule
bioactivities_api = new_client.activity

In [ ]:
# UniProt ID of target: Human Androgen Receptor
uniprot_id = "P10275"

# Fetch target data from ChEMBL
target_list = targets_api.get(target_components__accession=uniprot_id).only(
    "target_chembl_id", "organism", "pref_name", "target_type"
)

# Download target data into data frame
target_list = pd.DataFrame.from_records(target_list)
target_list

In [ ]:
# Select the first entry as our target of interest
target = target_list.iloc[0]
target

In [ ]:
# Save selected ChEMBL ID
chembl_id = target.target_chembl_id
print(f"The target ChEMBL ID is {chembl_id}")

In [ ]:
# Fetch bioactivity data from ChEMBL
# Filter it to only consider human proteins, bioactivity type IC50, exact measurements, and binding data
bioactivities = bioactivities_api.filter(
    target_chembl_id=chembl_id, type="IC50", relation="=", assay_type="B"
).only(
    "activity_id",
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "molecule_chembl_id",
    "type",
    "standard_units",
    "relation",
    "standard_value",
    "target_chembl_id",
    "target_organism",
)

print(f"Length and type of bioactivities object: {len(bioactivities)}, {type(bioactivities)}")

In [ ]:
# Example of information shown in each entry
print(f"Length and type of first element: {len(bioactivities[0])}, {type(bioactivities[0])}")
bioactivities[0]

In [ ]:
# Download bioactivity data from ChEMBL into data frame
bioactivities_df = pd.DataFrame.from_dict(bioactivities)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
bioactivities_df.head()


### 2. Data Curation and Cleaning
Real-world biological data is notoriously messy. A single target query can return data from hundreds of different labs using different methodologies. To prepare this dataset for subsequent processes, we must:
1. Convert IC50 data type to float.
2. Remove missing data (NaNs).
3. Ensure all binding affinities are measured in standard units (nM).
4. Drop duplicate tests for the same molecule.

In [ ]:
# Drop "units" and "value" columns and use only standardized units and values
bioactivities_df.drop(["units", "value"], axis=1, inplace=True)
bioactivities_df.head()

In [ ]:
# Convert "standard value" (IC50) into float
bioactivities_df = bioactivities_df.astype({"standard_value": "float64"})
bioactivities_df.dtypes

In [ ]:
# Remove entries with missing data
bioactivities_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
# Keep only entries with "standard units" = nM
bioactivities_df = bioactivities_df[bioactivities_df["standard_units"] == "nM"]
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
# Remove duplicate molecules
bioactivities_df.drop_duplicates("molecule_chembl_id", keep="first", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
# Reset index of the data frame
bioactivities_df.reset_index(drop=True, inplace=True)
bioactivities_df.head()

In [ ]:
# Rename the "standard_value" and "standard_units" columns into "IC50" and "units", respectively
bioactivities_df.rename(
    columns={"standard_value": "IC50", "standard_units": "units"}, inplace=True
)
bioactivities_df.head()

In [ ]:
print(f"DataFrame shape: {bioactivities_df.shape}")

### 3. Compound Structure Acquisition and Feature Extraction

Now that we have a clean dataset of biological activities (IC50 values), we are missing the physical structures of the molecules themselves. Machine learning models and pharmacophore software cannot read raw ChEMBL IDs; they require chemical representations.

In this phase, we will:
1. Fetch structural data from ChEMBL for all compounds in our filtered dataset.
2. Standardize and clean the data.
3. Extract Canonical SMILES strings for all compounds in our filtered dataset.

In [ ]:
# Fetch compound structural data from ChEMBL
compounds_provider = compounds_api.filter(
    molecule_chembl_id__in=list(bioactivities_df["molecule_chembl_id"])
).only("molecule_chembl_id", "molecule_structures")

In [ ]:
# Download compound data from ChEMBL
# Obtain list of records through tqdm as the data volume is large
compounds = list(tqdm(compounds_provider))

# Pass the list of compounds to the data frame
compounds_df = pd.DataFrame.from_records(
    compounds,
)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
compounds_df.head()

In [ ]:
# Remove entries with missing structural data
compounds_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
# Remove duplicate molecules
compounds_df.drop_duplicates("molecule_chembl_id", keep="first", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
# The data consists of multiple different molecular structure representations. We only want to keep the canonical SMILES
# Initialize an empty list to store the parsed SMILES strings
canonical_smiles = []

# Iterate over each row to extract the canonical SMILES from the nested dictionary
for i, compounds in compounds_df.iterrows():
    try:
        canonical_smiles.append(compounds["molecule_structures"]["canonical_smiles"])
    except KeyError:
        # If 'canonical_smiles' is missing, insert None to maintain row alignment
        canonical_smiles.append(None)

# Append the extracted list as a new, clean text column
compounds_df["smiles"] = canonical_smiles

# Drop the original nested dictionary column
compounds_df.drop("molecule_structures", axis=1, inplace=True)

print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
# Sanity check: remove molecules without canonical SMILES string
compounds_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

### 4. Data Integration and pIC50 Transformation

With our biological activity data (IC50) and chemical structures (SMILES) cleaned independently, our final step in data preparation is to fuse them into a single, machine-learning-ready master dataset. We will:

1. Merge datasets using the unique `molecule_chembl_id` to link each drug's chemical structure directly to its lab-tested biological activity.
2. Transform raw IC50 values to pIC50 which provides a normalized, linear scale where higher values indicate stronger drug potency—a much easier target for machine learning algorithms to predict.
3. Visual validation through generation of 2D molecular structures using RDKit, and plot the distribution of our new pIC50 scores.
4. Save the finalized dataset to a CSV file, completing our data acquisition pipeline and preparing us for ML model training.


In [ ]:
# Summary of bioactivity data
print(f"Bioactivities filtered: {bioactivities_df.shape[0]}")
print(bioactivities_df.columns)

In [ ]:
# Summary of compound data
print(f"Compounds filtered: {compounds_df.shape[0]}")
print(compounds_df.columns)

In [ ]:
# Merge the bioactivity data and the compound data
output_df = pd.merge(
    bioactivities_df[["molecule_chembl_id", "IC50", "units"]],
    compounds_df,
    on="molecule_chembl_id",
)

# Reset row indices
output_df.reset_index(drop=True, inplace=True)

print(f"Dataset with {output_df.shape[0]} entries.")

In [ ]:
output_df.head(10)

In [ ]:
# Define a function to convert raw IC50 values to pIC values
def convert_ic50_to_pic50(IC50_value):
    pIC50_value = 9 - math.log10(IC50_value)
    return pIC50_value

In [ ]:
# Apply conversion to each row of the compounds DataFrame
output_df["pIC50"] = output_df.apply(lambda x: convert_ic50_to_pic50(x.IC50), axis=1)
print(f"DataFrame shape: {output_df.shape}")

In [ ]:
output_df.head()

In [ ]:
# Plot a pIC50 value distribution using histogram
output_df.hist(column="pIC50")

In [ ]:
# Add a column for RDKit molecule objects to look at the structures of the molecules
PandasTools.AddMoleculeColumnToFrame(output_df, smilesCol="smiles")

# Render SVG pictures in our notebook
PandasTools.RenderImagesInAllDataFrames(images=True)

# Sort molecules by pIC50
output_df.sort_values(by="pIC50", ascending=False, inplace=True)

# Reset index
output_df.reset_index(drop=True, inplace=True)

In [ ]:
# Output only the structures of the molecules with the top 3 highest pIC50 values
output_df.drop("smiles", axis=1).head(3)

In [ ]:
# Drop the ROMol column for file saving
output_df = output_df.drop("ROMol", axis=1)
print(f"DataFrame shape: {output_df.shape}")

In [ ]:
# Save the data to csv file
output_df.to_csv(DATA / "AR_compounds.csv")
output_df.head()